In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    when,
    round
)

FACT_TABLE = "workspace.default.fact_trip"
PROVIDER_TABLE = "workspace.default.dim_provider"
DATE_TABLE = "workspace.default.dim_date"

df_fact = spark.table(FACT_TABLE)
df_provider = spark.table(PROVIDER_TABLE)
df_date = spark.table(DATE_TABLE)

print("Fact rows:", df_fact.count())
print("Provider rows:", df_provider.count())
print("Date rows:", df_date.count())

In [0]:
df_enriched = (
    df_fact
    .join(
        df_provider.select(
            "provider_key",
            "provider_name"
        ),
        on="provider_key",
        how="left"
    )
    .join(
        df_date.select(
            "date_key",
            "date"
        ),
        on="date_key",
        how="left"
    )
)

display(
    df_enriched.select(
        "date_key",
        "date",
        "provider_key",
        "provider_name",
        "trip_miles",
        "trip_time",
        "customer_wait_seconds",
        "base_passenger_fare",
        "tips",
        "driver_pay"
    ).limit(10)
)

In [0]:
df_gold = (
    df_enriched
    .groupBy(
        "date_key",
        "date",
        "provider_key",
        "provider_name"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("trip_miles").alias("total_trip_miles"),

        avg("trip_miles").alias("avg_trip_distance"),

        avg("calculated_trip_time_seconds").alias(
            "avg_trip_duration_seconds"
        ),

        avg("customer_wait_seconds").alias(
            "avg_customer_wait_seconds"
        ),

        sum("base_passenger_fare").alias(
            "total_passenger_fare"
        ),

        sum("tips").alias("total_tips"),

        sum("driver_pay").alias("total_driver_pay"),

        sum(
            when(
                col("overall_quality_status") != "GOOD",
                1
            ).otherwise(0)
        ).alias("quality_issue_trips")
    )
)

In [0]:
df_gold = (
    df_gold
    .withColumn(
        "avg_trip_duration_minutes",
        round(col("avg_trip_duration_seconds") / 60, 2)
    )
    .withColumn(
        "avg_customer_wait_minutes",
        round(col("avg_customer_wait_seconds") / 60, 2)
    )
)

In [0]:
df_gold = df_gold.drop(
    "avg_trip_duration_seconds",
    "avg_customer_wait_seconds"
)

In [0]:
df_gold = (
    df_gold
    .withColumn(
        "fare_per_mile",
        round(
            when(
                col("total_trip_miles") > 0,
                col("total_passenger_fare")
                / col("total_trip_miles")
            ),
            2
        )
    )
    .withColumn(
        "driver_pay_per_mile",
        round(
            when(
                col("total_trip_miles") > 0,
                col("total_driver_pay")
                / col("total_trip_miles")
            ),
            2
        )
    )
)

In [0]:
display(
    df_gold.orderBy(
        "date_key",
        "provider_key"
    )
)

In [0]:
print("Gold rows:", df_gold.count())

print(
    "Total fact rows:",
    df_fact.count()
)

print(
    "Gold total trips:",
    df_gold.select(
        sum("total_trips")
    ).collect()[0][0]
)

In [0]:
GOLD_TABLE = "workspace.default.gold_daily_provider_metrics"

(
    df_gold
    .write
    .format("Delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

In [0]:
display(df_gold.limit(10))

In [0]:
df_gold_final = spark.table(GOLD_TABLE)

print("Persisted rows:", df_gold_final.count())
print("Persisted columns:", len(df_gold_final.columns))

In [0]:
display(
    df_gold_final.orderBy(
        "date_key",
        "provider_key"
    )
)

In [0]:
print(
    "Persisted Gold trips:",
    df_gold_final
    .select(sum("total_trips"))
    .collect()[0][0]
)

In [0]:
import sys

SRC_PATH = "/Workspace/Users/gbpatil2002@gmail.com/nyc-hvfhv-data-platform/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from transformation.gold import build_daily_provider_gold

In [0]:
import importlib
import transformation.gold

importlib.reload(transformation.gold)

from transformation.gold import build_daily_provider_gold

In [0]:
build_daily_provider_gold(
    spark,
    "2026-02",
    "workspace.default.fact_trip",
    "workspace.default.dim_provider",
    "workspace.default.dim_date",
    "workspace.default.gold_daily_provider_metrics"
)

In [0]:
df_gold = spark.table(
    "workspace.default.gold_daily_provider_metrics"
)

print("Gold rows:", df_gold.count())
print("Gold columns:", len(df_gold.columns))

display(
    df_gold
    .orderBy("date", "provider_key")
    .limit(10)
)

In [0]:
from pyspark.sql.functions import month, year

feb_gold = (
    df_gold
    .filter(
        (year("date") == 2026) &
        (month("date") == 2)
    )
)

print("February Gold rows:", feb_gold.count())

In [0]:
from pyspark.sql import functions as F

print(
    "February Gold trips:",
    feb_gold
    .agg(F.sum("total_trips").alias("total_trips"))
    .first()["total_trips"]
)

In [0]:
df_gold = spark.table(
    "workspace.default.gold_daily_provider_metrics"
)

print("Total Gold rows:", df_gold.count())

print(
    "Total Gold trips:",
    df_gold
    .agg(F.sum("total_trips").alias("total_trips"))
    .first()["total_trips"]
)